# 05b — Request shapes on the pilot turns (Q28)

**Status:** ruled on 2026-09-28: shapes7, unchanged (D99), with requests split
by shape and slash command (D100). This notebook is the pilot evidence the
ruling was taken on. It puts the pilot's turns under every candidate rule of
`src/shapes.py`, beside the suite (`notebooks/sim/07_shapes`, run `42cab17a`).

The main question is whether `shapes7` is the right grain. It is unchanged here:
`get_weather` is `factual.forecast`, the two climate tools are
`factual.climate`, and the only exception is a search that feeds a draft.

Sections:

1. every pilot turn under every rule
2. label counts side by side
3. the `other` turns: which tools were mixed
4. forecast vs climate: do they appear together?
5. request boundaries (Q29): what segmentation does to the counts
6. pilot vs suite: which shapes the suite does not cover

**Scope.** Pilot sessions are technical tests by the author and a colleague,
excluded from analysis (`CLAUDE.md`). With this n the numbers are a check that
the rules behave on real traces, not an estimate of class sizes.

**Privacy.** Outputs are stripped on commit (`notebooks/study/.gitattributes`).
Nothing below prints a participant's words, an id or an email. Sessions are
`S1, …` in consent order (`PilotData.pseudonyms`) and Tonic runs `R1, …` in
order of first turn. Tool names, slash-command names and labels are the only
per-turn fields shown.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[1]))

from collections import Counter

import pandas as pd

from src import db, pilot, shapes, survey
from src.execution import _place_ids_of
from src.segmentation_bench import VARIANTS

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 90)

# --- parameters -------------------------------------------------------------
RUN_TYPE = "pilot"
SUITE_RUN_ID = "42cab17a-4903-4e39-b00c-ace76b7929e4"  # 07_shapes' run: the suite's shapes
RULES = list(shapes.RULES)

client = db.admin()
data = pilot.load(client, RUN_TYPE)
sessions = data.pseudonyms()
turns = list(data.suggestions)

replies = [t["reply_message_id"] for t in turns if t.get("reply_message_id")]
refs = {m["id"]: _place_ids_of(m.get("metadata"))
        for m in (client.table("chat_messages").select("id, metadata").in_("id", replies).execute().data
                  if replies else [])}

print(f"run_type={RUN_TYPE!r}: {len(data.sessions)} sessions "
      f"({sum(1 for s in data.sessions if s.get('consent_at'))} consented), {len(turns)} turns")
print(f"shape rule version {shapes.REQUEST_SHAPE_RULE_VERSION}")

## 1. Every pilot turn, under every rule

One row per agent turn. `acting tools` omits the three read tools, which every
rule ignores. `request_type` is the label the app stored. The `d09` column
re-derives it, and the two must agree; otherwise the comparison is between
two codebases rather than two rules.

In [ ]:
first_turn = {}
for t in sorted(turns, key=lambda t: t.get("created_at") or ""):
    first_turn.setdefault(t.get("run_id"), len(first_turn) + 1)

rows = []
for t in turns:
    tools = shapes.tools_of(t.get("tool_call_trace") or [])
    rows.append({
        "session": sessions.get(t.get("session_id"), "?"),
        "run": f"R{first_turn[t.get('run_id')]}",
        "turn": t.get("turn_index"),
        "slash": t.get("agent_intent") or "-",
        "acting tools": " + ".join(sorted(shapes._acting(tools))) or "-",
        "request_type": t.get("request_type"),
        **{rule: shapes.RULES[rule](tools) for rule in RULES},
        "_tools": tools,
    })
frame = pd.DataFrame(rows).sort_values(["run", "turn"]).reset_index(drop=True)

disagree = frame[frame["d09"] != frame["request_type"]]
print(f"d09 re-derived vs stored request_type: {len(disagree)} disagreement(s)")
frame.drop(columns="_tools")

## 2. Label counts, side by side

`shapes9` differs from `shapes7` only on live edits (move / place / remove). If
the pilot has none, the two columns are identical and the pilot says nothing
about the split.

In [ ]:
counts = pd.DataFrame({rule: frame[rule].value_counts() for rule in RULES}).fillna(0).astype(int)
counts.loc["(labels)"] = [frame[rule].nunique() for rule in RULES]
counts

## 3. The `other` turns

`shapes7` has no precedence, so a turn doing two things is `other`. That is the
rule working as designed, and Q28 recommends reporting `other` as its own row.
Whether a mix *recurs* is what matters for the decision. A mix seen more than
once is a shape the rule has no name for; a one-off is noise at this n.

In [ ]:
other = frame[frame["shapes7"] == shapes.OTHER]
print(f"shapes7 other: {len(other)} of {len(frame)} turns ({len(other) / max(len(frame), 1):.0%})")
(other.groupby("acting tools").size().rename("turns").sort_values(ascending=False).to_frame()
 if len(other) else "no other turns")

## 4. Forecast vs climate

`shapes7` keeps `factual.forecast` (`get_weather`) and `factual.climate`
(`get_climate_normals`, `get_region_climate`) apart, on the reading that a
forecast is a distinct call by the agent. That reading costs something only if
the two tools often appear **in the same turn**, which would put ordinary
weather questions into `other`. This counts how often each appears, alone,
together, and alongside other tools.

In [ ]:
FORECAST, CLIMATE = {"get_weather"}, {"get_climate_normals", "get_region_climate"}

def weather_kind(tools):
    s = set(tools)
    f, c = bool(s & FORECAST), bool(s & CLIMATE)
    if not (f or c):
        return None
    kind = "forecast + climate" if f and c else "forecast only" if f else "climate only"
    rest = shapes._acting(s) - FORECAST - CLIMATE
    return kind + (" (with " + " + ".join(sorted(rest)) + ")" if rest else "")

weather = frame["_tools"].map(weather_kind).dropna().rename("weather tools in the turn")
weather.value_counts().rename("turns").to_frame() if len(weather) else "no turn called a weather or climate tool"


## 5. Request boundaries (Q29)

A Tonic run can hold several requests. `shapes.segment_with` splits a run's
turns using the shape rule (`shapes7`), plus two optional signals: a slash
command starts a new request, and a reference to something the current request
offered continues it. "Follow-ups" are turns that did not open a request. Q30/D82
leave open whether P3 must score follow-ups, and this count feeds that.

Two request labels are shown. `interaction_label` is the current rule: a
request whose turns have two acting labels is `other`. The **union** labels a
request by all the tools its turns called, which makes a search-then-draft
across two turns read the same as within one (the question left open in Q29).

In [ ]:
runs = {}
for t in turns:
    runs.setdefault(t.get("run_id"), []).append(t)
runs = {k: sorted(v, key=lambda t: t.get("turn_index") or 0) for k, v in runs.items()}
print(f"{len(runs)} Tonic runs; turns per run: {sorted(len(v) for v in runs.values())}; "
      f"turns with a slash command: {sum(1 for t in turns if t.get('agent_intent'))}")

seg_rows, label_rows = [], []
for variant, (use_intent, use_reference) in VARIANTS.items():
    n_requests, by_turns, by_union = 0, Counter(), Counter()
    for run in runs.values():
        signals = [shapes.signals_of(t.get("tool_call_trace") or [], refs.get(t.get("reply_message_id"), ()),
                                     t.get("agent_intent"), shapes.shapes7) for t in run]
        segments = shapes.segment_with(signals, use_intent=use_intent, use_reference=use_reference)
        n_requests += len(segments)
        for seg in segments:
            by_turns[shapes.interaction_label([signals[i].label for i in seg])] += 1
            union = [n for i in seg for n in shapes.tools_of(run[i].get("tool_call_trace") or [])]
            by_union[shapes.shapes7(union)] += 1
    seg_rows.append({"variant": variant, "requests": n_requests, "follow-up turns": len(turns) - n_requests})
    label_rows += [{"variant": variant, "label by": "interaction_label", **by_turns},
                   {"variant": variant, "label by": "union of tools", **by_union}]

display(pd.DataFrame(seg_rows).set_index("variant"))
pd.DataFrame(label_rows).set_index(["variant", "label by"]).fillna(0).astype(int)

## 6. Pilot vs suite

The rule exists to say **where uncovered turns concentrate** (§4.3 (3)). This
puts the pilot's `shapes7` shares next to the suite's (run `42cab17a`, one row
per agent turn, as in `07_shapes`). A pilot label with no suite turn is a shape
the suite does not exercise.

In [ ]:
suite = survey.turn_frame(SUITE_RUN_ID)
suite["shapes7"] = suite["tool_names"].map(shapes.shapes7)

compare = pd.DataFrame({
    "suite turns": suite["shapes7"].value_counts(),
    "pilot turns": frame["shapes7"].value_counts(),
}).fillna(0).astype(int)
compare["suite share"] = (compare["suite turns"] / compare["suite turns"].sum()).map("{:.0%}".format)
compare["pilot share"] = (compare["pilot turns"] / compare["pilot turns"].sum()).map("{:.0%}".format)
compare["in suite"] = compare["suite turns"] > 0
compare.sort_values("pilot turns", ascending=False)

## For the ruling

What Q28 asks the author to settle, with the section that bears on each:

- **Grain:** `d09` vs `shapes7` vs `shapes9` vs `families`. See §2 for the
  number of labels and how thinly they spread, and §6 for what each would call
  uncovered.
- **Forecast kept distinct from climate:** §4. This only costs something if
  the two appear in one turn.
- **The draft exception:** only a search counts toward a draft. §3 shows
  whether another tool recurs with `propose_plan`.
- **`advisory` = called no tool** (sub-decision (i)): the `advisory` row in §2.
- **Request label** (open alongside Q29): the two label rows per variant in §5.

The ruling goes to `docs/DECISIONS.md` and freezes `REQUEST_SHAPE_RULE_VERSION`
at C05.